## Cell 1 — Imports

Imports all required libraries: PyTorch, torchvision, albumentations, scikit-learn, and Swin-Large backbone (via timm).

In [ ]:
import os
import sys
import random
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from pathlib import Path
from typing import Tuple, List, Dict, Optional, Any
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision
from torchvision import transforms

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight

# Import timm library for Swin-Large
try:
    import timm
    print("\u2713 Using timm library for Swin-Large")
    HAS_TIMM = True
except ImportError:
    print("\u26a0 Warning: timm is not available")
    print("Installing timm...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'timm'])
    import timm
    HAS_TIMM = True

warnings.filterwarnings('ignore')
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")


## Cell 2 — Seed and Device Setup

Sets global random seeds for reproducibility and selects CUDA/MPS/CPU device.

In [ ]:
def set_seed(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

DEVICE = torch.device(
    'cuda'  if torch.cuda.is_available()  else
    'mps'   if torch.backends.mps.is_available() else
    'cpu'
)
print(f"\u2705 Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Cell 3 — Configuration

Defines the `Config` class containing all hyperparameters and dataset paths.

In [ ]:
# CELL 3: CONFIGURATION PARAMETERS — Cross-Species Setup
# Train: Mango + Papaya  |  Test (held-out): Guava
# ============================================================================
class Config:
    """Central configuration for cross-species Swin-Large evaluation."""

    # Point DATA_ROOT at the raw (unsplit) dataset directory.
    # Expected structure:  DATA_ROOT / "Plant_Health" / image.jpg
    # e.g.  DATA_ROOT / "Guava_Healthy" / img001.jpg
    DATA_ROOT = Path("/workspace/Dataset_raw")   # <-- update this path
    OUTPUT_DIR = Path("outputs")
    CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
    LOG_DIR = OUTPUT_DIR / "logs"

    # Cross-species split: train on these, test on these (never mixed)
    TRAIN_PLANTS = ['Mango', 'Papaya']   # species seen during training
    TEST_PLANTS  = ['Guava']             # species held out entirely for test
    PLANTS       = ['Guava', 'Mango', 'Papaya']  # all species (reference only)

    CLASSES = ['Healthy', 'Anthracnose']
    SPECIES_CLASSES = ['mango', 'papaya']   # training species only
    HEALTH_CLASSES  = ['healthy', 'anthracnose']

    # num_species = 2 (only training species have a valid class index)
    NUM_SPECIES = 2
    NUM_HEALTH  = 2

    # No test split from training species — all guava is the test set.
    # Use 85/15 train/val within Mango + Papaya.
    TRAIN_RATIO = 0.85
    VAL_RATIO   = 0.15

    IMG_SIZE = 224
    IMG_MEAN = [0.485, 0.456, 0.406]
    IMG_STD  = [0.229, 0.224, 0.225]

    BATCH_SIZE    = 32
    NUM_WORKERS   = 0
    EPOCHS        = 100
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY  = 5e-4
    DROPOUT       = 0.4

    MIN_LR = 1e-7
    PATIENCE = 15
    SEED     = 42

    LOSS_WEIGHT_HEALTH = 1.0
    USE_AMP = True

    FPN_CHANNELS  = 128
    GCA_REDUCTION = 8
    PRETRAINED    = True

    AUG_PROB = 0.5

    @classmethod
    def create_dirs(cls):
        cls.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        cls.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
        cls.LOG_DIR.mkdir(parents=True, exist_ok=True)
        print("\u2705 Output directories created")

Config.create_dirs()


DATA_ROOT          = Config.DATA_ROOT
IMG_SIZE           = Config.IMG_SIZE
BATCH_SIZE         = Config.BATCH_SIZE
NUM_WORKERS        = Config.NUM_WORKERS
EPOCHS             = Config.EPOCHS
LR                 = Config.LEARNING_RATE
WEIGHT_DECAY       = Config.WEIGHT_DECAY
DROPOUT            = Config.DROPOUT
FPN_CHANNELS       = Config.FPN_CHANNELS
GCA_REDUCTION      = Config.GCA_REDUCTION
SEED               = Config.SEED
LOSS_WEIGHT_HEALTH = Config.LOSS_WEIGHT_HEALTH
USE_AMP            = Config.USE_AMP
PATIENCE           = Config.PATIENCE
NUM_SPECIES        = Config.NUM_SPECIES
NUM_HEALTH         = Config.NUM_HEALTH
PRETRAINED         = Config.PRETRAINED
NORMALIZE_MEAN     = Config.IMG_MEAN
NORMALIZE_STD      = Config.IMG_STD
CONFUSION_MATRIX_DPI = 150

# Species map covers TRAINING species only.
# Guava (test-only) gets species_id=0 as a dummy — it is never evaluated.
SPECIES_MAP = {"mango": 0, "papaya": 1}
HEALTH_MAP  = {"healthy": 0, "anthracnose": 1}

print("\n\U0001f4cb Configuration Summary (Cross-Species Evaluation):")
print(f"   - Train Species : {Config.TRAIN_PLANTS}  \u2192 split {Config.TRAIN_RATIO}/{Config.VAL_RATIO} (train/val)")
print(f"   - Test Species  : {Config.TEST_PLANTS}  (all images held out, never seen in training)")
print(f"   - Data Root     : {Config.DATA_ROOT}")
print(f"   - Image Size    : {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"   - Batch Size    : {Config.BATCH_SIZE}")
print(f"   - Learning Rate : {Config.LEARNING_RATE}")
print(f"   - Epochs        : {Config.EPOCHS}")
print(f"   - Dropout       : {Config.DROPOUT}")
print(f"   - Patience      : {Config.PATIENCE}")
print(f"   - FPN Channels  : {Config.FPN_CHANNELS}")
print(f"   - GCA Reduction : {Config.GCA_REDUCTION}")


## Cell 4 — Dataset Loader

Defines `DatasetLoader` for scanning the raw dataset folders and building a DataFrame, and `FlatSampleDataset` for per-fold K-Fold loaders.

In [ ]:
class DatasetLoader:
    """Load and organise the raw dataset into a DataFrame with multi-task labels.

    Expected directory structure (unsplit raw dataset):
        DATA_ROOT / Plant_Health / image.jpg
    e.g.
        DATA_ROOT / Guava_Healthy     / img001.jpg
        DATA_ROOT / Guava_Anthracnose / img002.jpg
        DATA_ROOT / Mango_Healthy     / img003.jpg
        ...
    """

    VALID_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

    def __init__(self, data_root: Path, plants: List[str], classes: List[str]):
        self.data_root = Path(data_root)
        self.plants    = plants
        self.classes   = classes

    def load_dataset(self) -> pd.DataFrame:
        """Scan all plant/class folders and return a DataFrame.

        Columns: image_path, plant, class, species_id, health_id, label
        where  label = health_id  (0 = Healthy, 1 = Anthracnose)
        """
        data = []
        for plant in self.plants:
            for cls in self.classes:
                folder_name = f"{plant}_{cls}"
                folder_path = self.data_root / folder_name

                if not folder_path.exists():
                    print(f"\u26a0\ufe0f  Warning: {folder_path} not found \u2014 skipping")
                    continue

                imgs = [str(p) for p in folder_path.iterdir()
                        if p.suffix.lower() in self.VALID_EXTS]

                sp_id = SPECIES_MAP.get(plant.lower(), 0)  # 0 = dummy for test-only species (Guava)
                he_id = HEALTH_MAP[cls.lower()]

                for img_path in imgs:
                    data.append({
                        'image_path': img_path,
                        'plant':      plant,
                        'class':      cls,
                        'species_id': sp_id,
                        'health_id':  he_id,
                        'label':      he_id,          # binary health label
                        'folder':     folder_name,
                    })

        df = pd.DataFrame(data)
        print(f"\u2705 Loaded {len(df)} images from {df['folder'].nunique()} folders")
        return df

    @staticmethod
    def compute_class_weights(labels: np.ndarray) -> torch.Tensor:
        """Compute class weights for imbalanced datasets."""
        cw = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
        return torch.tensor(cw, dtype=torch.float32)

    @staticmethod
    def get_sample_weights(labels: np.ndarray) -> np.ndarray:
        """Per-sample weights for WeightedRandomSampler."""
        class_counts = Counter(labels)
        total = len(labels)
        cw = {cls: total / cnt for cls, cnt in class_counts.items()}
        return np.array([cw[lbl] for lbl in labels])


class FlatSampleDataset(Dataset):
    """Wraps an explicit list of (path, species_id, health_id) tuples.
    Applies albumentations transforms after converting PIL to numpy.
    Used to build per-fold train/val loaders.
    """

    def __init__(self, samples: List[Tuple[str, int, int]], transform=None):
        self.samples   = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, sp_id, he_id = self.samples[idx]
        try:
            img = Image.open(path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {path}: {e}")
            img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))

        if self.transform is not None:
            img_np = np.array(img)                        # PIL \u2192 numpy HWC uint8
            img = self.transform(image=img_np)['image']   # albumentations call
        else:
            img = torch.from_numpy(np.array(img)).permute(2, 0, 1).float() / 255.0

        return img, torch.tensor(sp_id, dtype=torch.long), torch.tensor(he_id, dtype=torch.long)


print("\u2705 DatasetLoader and FlatSampleDataset defined")


## Cell 5 — Stratified Split

Loads raw dataset, computes statistics, performs 70/15/15 stratified split, saves CSVs, and builds train/val/test sample tuples.

In [ ]:
# CELL 5: CROSS-SPECIES DATA SPLIT
# ─────────────────────────────────────────────────────────────────────────────
# Train + Val : Mango + Papaya  (85% train / 15% val, stratified)
# Test        : ALL Guava       (zero guava images in training)
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

print("\n" + "="*80)
print("Cross-Species Data Split")
print("  Train/Val species : Mango + Papaya  (85 / 15 split)")
print("  Test species      : Guava           (100% held out)")
print("="*80)

# ── 1. Load Mango + Papaya (train + val pool) ─────────────────────────────
loader_trainval = DatasetLoader(Config.DATA_ROOT, Config.TRAIN_PLANTS, Config.CLASSES)
df_trainval = loader_trainval.load_dataset()

print("\n\U0001f4ca Mango + Papaya distribution:")
print(df_trainval.groupby(['plant', 'class']).size().unstack(fill_value=0))

# ── 2. Stratified 85/15 split within Mango + Papaya ──────────────────────
strat_col = df_trainval['plant'] + '_' + df_trainval['class']
train_df, val_df = train_test_split(
    df_trainval,
    test_size=Config.VAL_RATIO,
    stratify=strat_col,
    random_state=Config.SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"\n\U0001f4ca Split Statistics (Mango + Papaya):")
print(f"   Training  : {len(train_df):5d} samples ({len(train_df)/len(df_trainval)*100:.1f}%)")
print(f"   Validation: {len(val_df):5d} samples ({len(val_df)/len(df_trainval)*100:.1f}%)")

for name, sdf in [('Train', train_df), ('Val', val_df)]:
    dist = sdf['class'].value_counts()
    print(f"   {name:6s}: Healthy={dist.get('Healthy', 0):4d}, "
          f"Anthracnose={dist.get('Anthracnose', 0):4d}")

# ── 3. Load ALL Guava as test ─────────────────────────────────────────────
# species_id = 0 (dummy via SPECIES_MAP.get default) — NOT evaluated for species accuracy.
loader_test = DatasetLoader(Config.DATA_ROOT, Config.TEST_PLANTS, Config.CLASSES)
test_df = loader_test.load_dataset()

print(f"\n\U0001f4ca Test Set (Guava — never seen in training):")
print(test_df.groupby(['plant', 'class']).size().unstack(fill_value=0))
print(f"   Total: {len(test_df)} images")

# ── 4. Save splits for reproducibility ───────────────────────────────────
train_df.to_csv(Config.OUTPUT_DIR / 'train_split.csv', index=False)
val_df.to_csv(Config.OUTPUT_DIR   / 'val_split.csv',   index=False)
test_df.to_csv(Config.OUTPUT_DIR  / 'test_split.csv',  index=False)
print("\n\u2705 Splits saved to output directory")

# ── 5. Visualise distributions ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, sdf, color) in zip(axes, [
    ('Train (Mango+Papaya)',      train_df, 'steelblue'),
    ('Validation (Mango+Papaya)', val_df,   'orange'),
    ('Test (Guava — held out)',   test_df,  'tomato'),
]):
    dist = sdf.groupby(['plant', 'class']).size().reset_index(name='count')
    bars = ax.bar(
        dist['plant'] + '\n' + dist['class'],
        dist['count'],
        color=color, alpha=0.8, edgecolor='white'
    )
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_ylabel('Count')
    for bar in bars:
        ax.annotate(f'{int(bar.get_height())}',
                    (bar.get_x() + bar.get_width() / 2., bar.get_height()),
                    ha='center', va='bottom', fontsize=10)

plt.suptitle('Cross-Species Data Split', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(Config.OUTPUT_DIR / 'cross_species_split_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# ── 6. Convert to sample tuples ───────────────────────────────────────────
def _df_to_samples(sdf):
    return [(row.image_path, row.species_id, row.health_id)
            for _, row in sdf.iterrows()]

train_samples = _df_to_samples(train_df)
val_samples   = _df_to_samples(val_df)
test_samples  = _df_to_samples(test_df)

print(f"\n\u2705 train_samples : {len(train_samples)} images (Mango + Papaya)")
print(f"\u2705 val_samples   : {len(val_samples)} images (Mango + Papaya)")
print(f"\u2705 test_samples  : {len(test_samples)} images (Guava — unseen species)")


## Cell 6 — Augmentation Pipeline

Defines `AugmentationPipeline` using albumentations with 9 transforms for training and simple resize+normalize for validation. Sets `train_tf` and `eval_tf` used by `FlatSampleDataset`.

In [ ]:
class AugmentationPipeline:
    """Comprehensive augmentation pipeline using albumentations."""

    @staticmethod
    def get_train_transforms(img_size: int = 224, aug_prob: float = 0.5) -> A.Compose:
        return A.Compose([
            # 1. Random Rotation (\u00b130 degrees)
            A.Rotate(limit=30, p=aug_prob, border_mode=cv2.BORDER_REFLECT_101),
            # 2. Horizontal and Vertical Flip
            A.HorizontalFlip(p=aug_prob),
            A.VerticalFlip(p=aug_prob * 0.5),
            # 3. Brightness and Contrast Jitter
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=aug_prob),
            # 4. Random Resized Crop
            A.RandomResizedCrop(
                size=(img_size, img_size), scale=(0.8, 1.0), ratio=(0.9, 1.1), p=aug_prob
            ),
            # 5. Color Jitter
            A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.2, hue=0.1, p=aug_prob),
            # 6. Gaussian Blur
            A.GaussianBlur(blur_limit=(3, 7), p=aug_prob * 0.3),
            # 7. Random Shadow
            A.RandomShadow(
                shadow_roi=(0, 0.5, 1, 1),
                num_shadows_lower=1, num_shadows_upper=2,
                shadow_dimension=5, p=aug_prob * 0.3
            ),
            # 8. CLAHE
            A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=aug_prob * 0.3),
            # 9. Coarse Dropout
            A.CoarseDropout(
                max_holes=8,
                max_height=img_size // 8, max_width=img_size // 8,
                min_holes=1,
                min_height=img_size // 16, min_width=img_size // 16,
                fill_value=0, p=aug_prob * 0.3
            ),
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    @staticmethod
    def get_val_transforms(img_size: int = 224) -> A.Compose:
        return A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

    @staticmethod
    def get_visualization_transforms(img_size: int = 224) -> A.Compose:
        """Transforms for visualization (without normalization)."""
        return A.Compose([
            A.Rotate(limit=30, p=1.0, border_mode=cv2.BORDER_REFLECT_101),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
            A.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0), ratio=(0.9, 1.1), p=1.0),
            A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.2, hue=0.1, p=1.0),
            A.GaussianBlur(blur_limit=(3, 5), p=0.3),
            A.CoarseDropout(max_holes=4, max_height=20, max_width=20, p=0.3),
            A.Resize(img_size, img_size),
        ])

# Create transform instances used by FlatSampleDataset
train_tf  = AugmentationPipeline.get_train_transforms(Config.IMG_SIZE, Config.AUG_PROB)
eval_tf   = AugmentationPipeline.get_val_transforms(Config.IMG_SIZE)
vis_transforms = AugmentationPipeline.get_visualization_transforms(Config.IMG_SIZE)

print("\u2705 Augmentation pipelines created")
print(f"   train_tf : {len(train_tf)}  transforms (with augmentation)")
print(f"   eval_tf  : {len(eval_tf)} transforms (resize + normalize only)")


## Cell 7 — Visualise Augmentations

Displays a grid of augmented versions of sample images from each class.

In [ ]:
def visualize_augmentations(
        samples: List[Tuple[str, int, int]],
        vis_transform: A.Compose,
        num_samples: int = 3,
        num_augmentations: int = 5,
        save_path: Optional[Path] = None):
    """Visualise augmentation effects on sample images from each species/health combo."""
    REVERSE_SP = {v: k for k, v in SPECIES_MAP.items()}
    REVERSE_HE = {v: k for k, v in HEALTH_MAP.items()}

    # Group samples by (species, health)
    from collections import defaultdict
    groups = defaultdict(list)
    for path, sp, he in samples:
        groups[(sp, he)].append(path)

    class_keys = sorted(groups.keys())
    total_rows = len(class_keys) * num_samples
    fig, axes = plt.subplots(
        total_rows, num_augmentations + 1,
        figsize=(3 * (num_augmentations + 1), 3 * total_rows)
    )
    if total_rows == 1:
        axes = axes[np.newaxis, :]

    row_idx = 0
    for (sp, he) in class_keys:
        paths = groups[(sp, he)]
        chosen = random.sample(paths, min(num_samples, len(paths)))
        label_str = f"{REVERSE_SP[sp]} / {REVERSE_HE[he]}"

        for img_path in chosen:
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (Config.IMG_SIZE, Config.IMG_SIZE))

            axes[row_idx, 0].imshow(img)
            axes[row_idx, 0].set_title(f'Original\n({label_str})', fontsize=9)
            axes[row_idx, 0].axis('off')

            for aug_idx in range(num_augmentations):
                augmented = vis_transform(image=img)['image']
                axes[row_idx, aug_idx + 1].imshow(augmented)
                axes[row_idx, aug_idx + 1].set_title(f'Aug {aug_idx + 1}', fontsize=9)
                axes[row_idx, aug_idx + 1].axis('off')

            row_idx += 1

    plt.suptitle('Augmentation Pipeline Visualisation\n(5 Augmentations per Sample)',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"\u2705 Visualisation saved to {save_path}")

    plt.show()


print("\U0001f504 Visualising augmentation effects...")
visualize_augmentations(
    train_samples + val_samples + test_samples,
    vis_transforms,
    num_samples=2,
    num_augmentations=5,
    save_path=Config.OUTPUT_DIR / 'augmentation_visualization.png'
)


## Cell 8 — Demonstrate Augmentations

Shows each of the key augmentations applied individually to a sample image.

In [ ]:
def demonstrate_individual_augmentations(image_path: str, save_path: Optional[Path] = None):
    """Demonstrate each of the 5 main augmentations individually."""
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (Config.IMG_SIZE, Config.IMG_SIZE))

    augmentations = {
        'Original': A.Compose([A.Resize(Config.IMG_SIZE, Config.IMG_SIZE)]),
        '1. Rotation (\u00b130\u00b0)': A.Compose([
            A.Rotate(limit=30, p=1.0, border_mode=cv2.BORDER_REFLECT_101),
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE)
        ]),
        '2. H/V Flip': A.Compose([
            A.HorizontalFlip(p=1.0),
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE)
        ]),
        '3. Brightness/Contrast': A.Compose([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=1.0),
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE)
        ]),
        '4. Random Crop': A.Compose([
            A.RandomResizedCrop(size=(Config.IMG_SIZE, Config.IMG_SIZE), scale=(0.7, 0.9), p=1.0)
        ]),
        '5. Color Jitter': A.Compose([
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.3, hue=0.15, p=1.0),
            A.Resize(Config.IMG_SIZE, Config.IMG_SIZE)
        ]),
    }

    fig, axes = plt.subplots(2, 3, figsize=(12, 8))
    axes = axes.flatten()

    for idx, (name, transform) in enumerate(augmentations.items()):
        augmented = transform(image=img)['image']
        axes[idx].imshow(augmented)
        axes[idx].set_title(name, fontsize=11, fontweight='bold')
        axes[idx].axis('off')

    plt.suptitle('5 Key Augmentations Demonstrated', fontsize=14, fontweight='bold')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Demonstrate on a sample image from the pool
sample_img_path = train_samples[0][0]
demonstrate_individual_augmentations(
    sample_img_path,
    save_path=Config.OUTPUT_DIR / 'individual_augmentations.png'
)


## Cell 9 — Anthracnose Dataset

Defines `AnthracnoseDataset`, the PyTorch `Dataset` subclass that loads images from a list of samples and returns `(image, species_label, health_label)` for multi-task learning.

In [ ]:
class AnthracnoseDataset(Dataset):
    """
    Custom dataset for Anthracnose multi-task classification.
    Returns (image_tensor, species_label, health_label).
    Accepts albumentations transforms \u2014 applies them after PIL\u2192numpy conversion.
    """

    def __init__(self,
                 samples: List[Tuple[str, int, int]],
                 transform: Optional[A.Compose] = None):
        """
        Args:
            samples : list of (image_path, species_id, health_id)
            transform: albumentations Compose pipeline
        """
        self.samples   = samples
        self.transform = transform

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, sp_id, he_id = self.samples[idx]

        # Load image as numpy array (albumentations expects HWC uint8)
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform is not None:
            img = self.transform(image=img)['image']
        else:
            img = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        return img, torch.tensor(sp_id, dtype=torch.long), torch.tensor(he_id, dtype=torch.long)


print("\u2705 AnthracnoseDataset (multi-task) defined")


## Cell 10 — Create Dataloaders

Defines `create_dataloaders()` and creates train, validation, and test data loaders.

In [ ]:
def create_dataloaders(
        train_samples: List[Tuple[str, int, int]],
        val_samples:   List[Tuple[str, int, int]],
        test_samples:  List[Tuple[str, int, int]],
        batch_size: int,
        sample_weights: Optional[np.ndarray] = None,
        num_workers: int = 0
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """Create data loaders; optionally uses WeightedRandomSampler for training."""
    train_ds = AnthracnoseDataset(train_samples, transform=train_tf)
    val_ds   = AnthracnoseDataset(val_samples,   transform=eval_tf)
    test_ds  = AnthracnoseDataset(test_samples,  transform=eval_tf)

    if sample_weights is not None:
        sampler = WeightedRandomSampler(
            weights=sample_weights, num_samples=len(sample_weights), replacement=True
        )
        train_loader = DataLoader(
            train_ds, batch_size=batch_size, sampler=sampler,
            num_workers=num_workers, pin_memory=(DEVICE.type == "cuda"), drop_last=True
        )
    else:
        train_loader = DataLoader(
            train_ds, batch_size=batch_size, shuffle=True,
            num_workers=num_workers, pin_memory=(DEVICE.type == "cuda"), drop_last=True
        )

    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=(DEVICE.type == "cuda")
    )
    test_loader = DataLoader(
        test_ds, batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=(DEVICE.type == "cuda")
    )
    return train_loader, val_loader, test_loader

train_loader, val_loader, test_loader = create_dataloaders(
    train_samples=train_samples,
    val_samples=val_samples,
    test_samples=test_samples,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

print(f"\u2705 train_loader: {len(train_loader)} batches ({len(train_samples)} images)")
print(f"\u2705 val_loader  : {len(val_loader)} batches ({len(val_samples)} images)")
print(f"\u2705 test_loader : {len(test_loader)} batches ({len(test_samples)} images)")
print(f"\u2705 train_tf / eval_tf  : albumentations pipelines ready")

## MODEL DEFINITION

In [ ]:
class GlobalContextAttention(nn.Module):
    """
    Global Context Attention Block for refining feature representations.

    This module captures long-range dependencies and global context,
    which is crucial for identifying disease patterns across the entire leaf.

    Architecture:
    - Global Average Pooling to capture global context
    - Channel attention via squeeze-excitation mechanism
    - Spatial attention for locating disease regions
    - Feature refinement through residual connection
    """

    def __init__(self, in_channels: int, reduction: int = 16):
        """
        Args:
            in_channels: Number of input channels
            reduction: Reduction ratio for squeeze-excitation
        """
        super().__init__()

        # Channel attention (Squeeze-Excitation)
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(in_channels, in_channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(in_channels // reduction, in_channels, bias=False),
            nn.Sigmoid()
        )

        # Spatial attention
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels // reduction),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction, 1, kernel_size=1, bias=False),
            nn.Sigmoid()
        )

        # Global context modeling
        self.global_context = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

        # Layer normalization for stability
        self.norm = nn.LayerNorm([in_channels])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of GCA block.

        Args:
            x: Input tensor of shape (B, C, H, W)

        Returns:
            Refined feature tensor of shape (B, C, H, W)
        """
        batch_size, channels, height, width = x.shape

        # Channel attention
        channel_weights = self.channel_attention(x)
        channel_weights = channel_weights.view(batch_size, channels, 1, 1)
        channel_refined = x * channel_weights

        # Spatial attention
        spatial_weights = self.spatial_attention(x)
        spatial_refined = x * spatial_weights

        # Combine channel and spatial attention
        combined = channel_refined + spatial_refined

        # Global context
        global_ctx = self.global_context(combined)

        # Residual connection
        output = x + global_ctx

        return output


class MultiHeadGCA(nn.Module):
    """
    Multi-head Global Context Attention for richer feature refinement.
    Uses multiple attention heads to capture different aspects of the disease.
    """

    def __init__(self, in_channels: int, num_heads: int = 4, reduction: int = 16):
        super().__init__()

        assert in_channels % num_heads == 0, "in_channels must be divisible by num_heads"

        self.num_heads = num_heads
        self.head_dim = in_channels // num_heads

        self.heads = nn.ModuleList([
            GlobalContextAttention(self.head_dim, reduction)
            for _ in range(num_heads)
        ])

        self.fusion = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        batch_size, channels, height, width = x.shape

        # Split into heads
        x_split = x.chunk(self.num_heads, dim=1)

        # Apply attention to each head
        attended = [head(x_i) for head, x_i in zip(self.heads, x_split)]

        # Concatenate and fuse
        output = torch.cat(attended, dim=1)
        output = self.fusion(output)

        return output


# ─────────────────────────────────────────────────────────────────────────────
# Swin-Large FPN Backbone
# ─────────────────────────────────────────────────────────────────────────────

class SwinLargeFPNBackbone(nn.Module):
    """
    Swin-Large backbone with Feature Pyramid Network.

    Swin-Large is hierarchical — it produces 4 genuine multi-scale spatial maps:
      Stage 1: (B, 192,  56x56)
      Stage 2: (B, 384,  28x28)
      Stage 3: (B, 768,  14x14)
      Stage 4: (B, 1536,  7x7)

    Hooks capture each stage output via timm's model.layers[0-3].
    The top-down FPN pathway provides genuine multi-scale feature fusion.
    """

    # Swin-Large channel dims per stage
    STAGE_DIMS = [192, 384, 768, 1536]

    def __init__(self, swin_model, fpn_channels: int = 256):
        super().__init__()
        self.swin = swin_model
        self.fpn_channels = fpn_channels
        self._feature_maps: Dict[int, torch.Tensor] = {}
        self._hooks = []
        self._register_hooks()

        # Lateral connections — each stage has a different channel count
        self.lateral1 = nn.Conv2d(self.STAGE_DIMS[0], fpn_channels, 1)
        self.lateral2 = nn.Conv2d(self.STAGE_DIMS[1], fpn_channels, 1)
        self.lateral3 = nn.Conv2d(self.STAGE_DIMS[2], fpn_channels, 1)
        self.lateral4 = nn.Conv2d(self.STAGE_DIMS[3], fpn_channels, 1)

        # Smooth layers
        self.smooth1 = nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1)
        self.smooth2 = nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1)
        self.smooth3 = nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1)
        self.smooth4 = nn.Conv2d(fpn_channels, fpn_channels, 3, padding=1)

        self.out_channels = fpn_channels

    def _get_stages(self):
        """Return the 4 stage modules from timm Swin."""
        if hasattr(self.swin, 'layers'):
            return list(self.swin.layers)
        elif hasattr(self.swin, 'features'):
            # torchvision fallback
            return [self.swin.features[1], self.swin.features[3],
                    self.swin.features[5], self.swin.features[7]]
        return None

    def _register_hooks(self):
        """Register forward hooks on each of the 4 Swin stages."""
        stages = self._get_stages()
        if stages is None:
            raise RuntimeError("Cannot find Swin stages in model.")
        for stage_idx, stage in enumerate(stages):
            hook = stage.register_forward_hook(
                lambda m, inp, out, idx=stage_idx: self._feature_maps.update({idx: out})
            )
            self._hooks.append(hook)

    def _to_nchw(self, feat: torch.Tensor) -> torch.Tensor:
        """
        Convert NHWC tensor (B, H, W, C) to NCHW (B, C, H, W) if needed.
        Detected when last dim > dim 1 (channels larger than spatial height).
        """
        if feat.dim() == 4 and feat.shape[-1] > feat.shape[1]:
            return feat.permute(0, 3, 1, 2).contiguous()
        return feat

    def _upsample_add(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        _, _, h, w = y.size()
        return F.interpolate(x, size=(h, w), mode='bilinear', align_corners=False) + y

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        self._feature_maps = {}

        # Forward through Swin — hooks capture each stage output
        _ = self.swin(x)

        # Convert NHWC → NCHW if needed
        c1 = self._to_nchw(self._feature_maps[0])  # stage 1: 56x56, 192ch
        c2 = self._to_nchw(self._feature_maps[1])  # stage 2: 28x28, 384ch
        c3 = self._to_nchw(self._feature_maps[2])  # stage 3: 14x14, 768ch
        c4 = self._to_nchw(self._feature_maps[3])  # stage 4:  7x7, 1536ch

        # FPN top-down pathway (genuine multi-scale for Swin)
        p4 = self.lateral4(c4)
        p3 = self._upsample_add(p4, self.lateral3(c3))
        p2 = self._upsample_add(p3, self.lateral2(c2))
        p1 = self._upsample_add(p2, self.lateral1(c1))

        # Smooth
        p4 = self.smooth4(p4)
        p3 = self.smooth3(p3)
        p2 = self.smooth2(p2)
        p1 = self.smooth1(p1)

        return {'p1': p1, 'p2': p2, 'p3': p3, 'p4': p4}


# ─────────────────────────────────────────────────────────────────────────────
# Multi-Task Swin-Large with FPN + GCA
# ─────────────────────────────────────────────────────────────────────────────

class MultiTaskSwinLarge(nn.Module):
    def __init__(self, num_species=NUM_SPECIES, num_health=NUM_HEALTH,
                 pretrained=PRETRAINED, dropout=DROPOUT,
                 fpn_channels=FPN_CHANNELS, gca_reduction=GCA_REDUCTION):
        super().__init__()

        # ── Load Swin-Large backbone (timm only) ──────────────────────────────
        import timm
        if pretrained:
            model_name = 'swin_large_patch4_window7_224.ms_in22k_ft_in1k'
        else:
            model_name = 'swin_large_patch4_window7_224'
        swin_model = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        print(f"\u2713 Using Swin-Large from timm library (model: {model_name})")

        # ── FPN Backbone (wraps Swin-Large with Feature Pyramid Network) ──────
        self.backbone = SwinLargeFPNBackbone(swin_model, fpn_channels=fpn_channels)

        # ── Multi-Head GCA applied to each FPN level ──────────────────────────
        self.gca = MultiHeadGCA(fpn_channels, num_heads=4, reduction=gca_reduction)

        # ── Feature fusion (4 FPN levels → fpn_channels) ─────────────────────
        self.feature_fusion = nn.Sequential(
            nn.Conv2d(fpn_channels * 4, fpn_channels * 2, kernel_size=1, bias=False),
            nn.BatchNorm2d(fpn_channels * 2),
            nn.ReLU(inplace=True),
            nn.Conv2d(fpn_channels * 2, fpn_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(fpn_channels),
            nn.ReLU(inplace=True)
        )

        # ── Global pooling ────────────────────────────────────────────────────
        self.global_pool = nn.AdaptiveAvgPool2d(1)

        # ── Multi-task classification heads ───────────────────────────────────
        self.dropout = nn.Dropout(dropout)
        self.head_species = nn.Linear(fpn_channels, num_species)
        self.head_health  = nn.Linear(fpn_channels, num_health)

        # ── Initialise new (non-pretrained) layers ────────────────────────────
        self._init_new_weights()

    def _init_new_weights(self):
        """Initialise weights for all newly added layers (FPN, GCA, heads)."""
        new_conv_layers = [
            self.backbone.lateral1, self.backbone.lateral2,
            self.backbone.lateral3, self.backbone.lateral4,
            self.backbone.smooth1,  self.backbone.smooth2,
            self.backbone.smooth3,  self.backbone.smooth4,
        ]
        for m in new_conv_layers:
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        for m in self.feature_fusion.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
        for linear in [self.head_species, self.head_health]:
            nn.init.xavier_normal_(linear.weight)
            nn.init.constant_(linear.bias, 0)

    def forward(self, x: torch.Tensor):
        # Extract multi-scale features via Swin-Large FPN
        features = self.backbone(x)

        # Apply GCA to each FPN level
        p1_att = self.gca(features['p1'])
        p2_att = self.gca(features['p2'])
        p3_att = self.gca(features['p3'])
        p4_att = self.gca(features['p4'])

        # Resize all FPN levels to match p4 spatial size (7x7), then concatenate
        target_size = p4_att.shape[-2:]
        p1_resized = F.adaptive_avg_pool2d(p1_att, target_size)
        p2_resized = F.adaptive_avg_pool2d(p2_att, target_size)
        p3_resized = F.adaptive_avg_pool2d(p3_att, target_size)

        multi_scale = torch.cat([p1_resized, p2_resized, p3_resized, p4_att], dim=1)
        fused = self.feature_fusion(multi_scale)

        # Global pooling → multi-task heads
        pooled = self.global_pool(fused).flatten(1)
        pooled = self.dropout(pooled)

        logits_species = self.head_species(pooled)
        logits_health  = self.head_health(pooled)

        return logits_species, logits_health

print("\u2713 Model class defined (instantiated fresh per fold)")


##  OPTIMIZER, SCHEDULER, LOSS FUNCTIONS

In [ ]:
criterion_species = nn.CrossEntropyLoss()
criterion_health  = nn.CrossEntropyLoss()
# scaler is reassigned as a global inside train_fold() for each fold
scaler = None
print("✓ Loss functions initialized")

## TRAINING UTILITIES

In [ ]:
def accuracy(logits, targets):
    """Calculate accuracy from logits and targets"""
    preds = logits.argmax(dim=1)
    return (preds == targets).float().mean().item()

def run_epoch(loader, model, optimizer=None, train=True, epoch=0):
    """Run one epoch of training or evaluation"""
    if train:
        model.train()
    else:
        model.eval()
    
    running = {
        "loss": 0.0,
        "acc_species": 0.0,
        "acc_health": 0.0,
        "acc_both": 0.0,
        "n": 0
    }
    total_batches = len(loader)
    
    for batch_idx, (imgs, y_species, y_health) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        y_species = y_species.to(device, non_blocking=True)
        y_health = y_health.to(device, non_blocking=True)
        
        with torch.set_grad_enabled(train):
            if USE_AMP and device.type == "cuda":
                with torch.amp.autocast('cuda'):
                    logits_species, logits_health = model(imgs)
                    loss = criterion_species(logits_species, y_species) + \
                           LOSS_WEIGHT_HEALTH * criterion_health(logits_health, y_health)
            else:
                logits_species, logits_health = model(imgs)
                loss = criterion_species(logits_species, y_species) + \
                       LOSS_WEIGHT_HEALTH * criterion_health(logits_health, y_health)
        
        if train:
            optimizer.zero_grad(set_to_none=True)
            if USE_AMP and device.type == "cuda":
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()
        
        # Calculate accuracies
        acc_sp = accuracy(logits_species, y_species)
        acc_he = accuracy(logits_health, y_health)
        preds_sp = logits_species.argmax(1)
        preds_he = logits_health.argmax(1)
        acc_both = ((preds_sp == y_species) & (preds_he == y_health)).float().mean().item()
        
        # Update running statistics
        bs = imgs.size(0)
        running["loss"] += loss.item() * bs
        running["acc_species"] += acc_sp * bs
        running["acc_health"] += acc_he * bs
        running["acc_both"] += acc_both * bs
        running["n"] += bs
        
        # Print progress
        if (batch_idx + 1) % max(1, total_batches // 10) == 0 or (batch_idx + 1) == total_batches:
            avg_loss = running["loss"] / running["n"]
            avg_sp = running["acc_species"] / running["n"]
            avg_he = running["acc_health"] / running["n"]
            avg_both = running["acc_both"] / running["n"]
            print(f"  [{batch_idx + 1}/{total_batches}] loss: {avg_loss:.4f}, "
                  f"sp: {avg_sp:.3f}, he: {avg_he:.3f}, both: {avg_both:.3f}")
    
    # Calculate averages
    for k in ["loss", "acc_species", "acc_health", "acc_both"]:
        running[k] /= max(1, running["n"])
    
    return running

## TRAINING LOOP

In [ ]:
import glob as _glob
import os
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)

def find_latest_epoch_ckpt():
    """Return the epoch_*.pt with the highest epoch number, or None."""
    files = _glob.glob("epoch_*.pt")
    if not files:
        return None
    def extract_epoch(fn):
        try:
            return int(fn.split("epoch_")[1].replace(".pt", ""))
        except Exception:
            return -1
    files.sort(key=extract_epoch)
    return files[-1]


def train_model():
    """Train the model with full checkpoint-resume support."""
    global scaler

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(SEED)

    print(f"\n{'='*80}")
    print(f"Training | Train: {len(train_samples)} | Val: {len(val_samples)}")
    print(f"{'='*80}")

    model     = MultiTaskSwinLarge(num_species=NUM_SPECIES, num_health=NUM_HEALTH,
                             pretrained=PRETRAINED, dropout=DROPOUT).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP and device.type == "cuda")

    start_epoch       = 0
    best_metric       = 0.0
    epochs_no_improve = 0
    history = {
        "train_loss": [], "val_loss": [],
        "train_acc_species": [], "val_acc_species": [],
        "train_acc_health":  [], "val_acc_health":  [],
        "train_acc_both":    [], "val_acc_both":    []
    }

    resume_ckpt = find_latest_epoch_ckpt()
    if resume_ckpt is not None:
        print(f"Resuming from checkpoint: {resume_ckpt}")
        ckpt = torch.load(resume_ckpt, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch       = ckpt["epoch"] + 1
        best_metric       = ckpt["best_metric"]
        epochs_no_improve = ckpt["epochs_no_improve"]
        history           = ckpt["history"]
        print(f"Resumed from epoch {ckpt['epoch']+1}, best_metric={best_metric:.4f}")

    for epoch in range(start_epoch, EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS} | LR: {optimizer.param_groups[0]['lr']:.2e}")

        train_stats = run_epoch(train_loader, model, optimizer, train=True,  epoch=epoch)
        val_stats   = run_epoch(val_loader,   model, optimizer=None, train=False, epoch=epoch)
        scheduler.step()

        history["train_loss"].append(train_stats["loss"])
        history["val_loss"].append(val_stats["loss"])
        history["train_acc_species"].append(train_stats["acc_species"])
        history["val_acc_species"].append(val_stats["acc_species"])
        history["train_acc_health"].append(train_stats["acc_health"])
        history["val_acc_health"].append(val_stats["acc_health"])
        history["train_acc_both"].append(train_stats["acc_both"])
        history["val_acc_both"].append(val_stats["acc_both"])

        print(f"  Train - Loss: {train_stats['loss']:.4f} | Sp: {train_stats['acc_species']:.3f} | He: {train_stats['acc_health']:.3f}")
        print(f"  Val   - Loss: {val_stats['loss']:.4f} | Sp: {val_stats['acc_species']:.3f} | He: {val_stats['acc_health']:.3f}")

        val_metric = (val_stats["acc_species"] + val_stats["acc_health"]) / 2.0

        if val_metric > best_metric:
            best_metric = val_metric
            epochs_no_improve = 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "val_metric": best_metric},
                       "best_model.pt")
            print(f"  \u2605 New best (val_metric={best_metric:.4f})")
        else:
            epochs_no_improve += 1
            print(f"  No improvement ({epochs_no_improve}/{PATIENCE})")

        prev_ckpt = find_latest_epoch_ckpt()
        epoch_ckpt_path = f"epoch_{epoch}.pt"
        torch.save({
            "model":             model.state_dict(),
            "optimizer":         optimizer.state_dict(),
            "scheduler":         scheduler.state_dict(),
            "epoch":             epoch,
            "best_metric":       best_metric,
            "epochs_no_improve": epochs_no_improve,
            "history":           history
        }, epoch_ckpt_path)
        if prev_ckpt is not None and prev_ckpt != epoch_ckpt_path:
            try:
                os.remove(prev_ckpt)
            except FileNotFoundError:
                pass

        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping after {PATIENCE} epochs without improvement.")
            break

    # Evaluate on guava test set — health only (species_id is dummy for guava)
    best_ckpt = torch.load("best_model.pt", map_location=device)
    model.load_state_dict(best_ckpt["model"])
    model.eval()
    all_he_preds, all_he_true = [], []
    with torch.no_grad():
        for imgs, _, y_he in test_loader:
            imgs = imgs.to(device, non_blocking=True)
            _, logits_he = model(imgs)
            all_he_preds.extend(logits_he.argmax(1).cpu().numpy())
            all_he_true.extend(y_he.numpy())

    task_b_acc       = accuracy_score(all_he_true, all_he_preds)
    health_f1        = f1_score(all_he_true, all_he_preds, average='weighted', zero_division=0)
    health_precision = precision_score(all_he_true, all_he_preds, average='weighted', zero_division=0)
    health_recall    = recall_score(all_he_true, all_he_preds, average='weighted', zero_division=0)

    print(f"\nGuava Test (cross-species) \u2014 Health Acc: {task_b_acc:.4f} | F1: {health_f1:.4f} | P: {health_precision:.4f} | R: {health_recall:.4f}")
    print(f"  Species Acc: N/A (Guava never seen in training)")

    pd.DataFrame(history).to_csv("training_history.csv", index=False)

    return {
        "task_a_accuracy":  None,          # N/A for cross-species evaluation
        "task_b_accuracy":  float(task_b_acc),
        "health_f1":        float(health_f1),
        "health_precision": float(health_precision),
        "health_recall":    float(health_recall),
    }


## COMPREHENSIVE TESTING FUNCTION

In [ ]:
def comprehensive_test(model, test_loader, device, species_map, health_map,
                       cross_species_test=False, test_species_name="Guava"):
    """
    Perform comprehensive testing with metrics and visualizations.

    cross_species_test: if True, species accuracy is suppressed (test species
    was never in training) and only health metrics are reported as primary results.
    """
    model.eval()

    all_species_preds = []
    all_species_true = []
    all_health_preds = []
    all_health_true = []
    all_both_correct = []

    print("Running comprehensive test evaluation...")

    with torch.no_grad():
        for batch_idx, (imgs, y_species, y_health) in enumerate(test_loader):
            imgs = imgs.to(device, non_blocking=True)
            y_species = y_species.to(device, non_blocking=True)
            y_health = y_health.to(device, non_blocking=True)

            logits_species, logits_health = model(imgs)
            preds_species = logits_species.argmax(dim=1)
            preds_health = logits_health.argmax(dim=1)

            all_species_preds.extend(preds_species.cpu().numpy())
            all_species_true.extend(y_species.cpu().numpy())
            all_health_preds.extend(preds_health.cpu().numpy())
            all_health_true.extend(y_health.cpu().numpy())

            both_correct = ((preds_species == y_species) & (preds_health == y_health)).cpu().numpy()
            all_both_correct.extend(both_correct)

    all_species_preds = np.array(all_species_preds)
    all_species_true  = np.array(all_species_true)
    all_health_preds  = np.array(all_health_preds)
    all_health_true   = np.array(all_health_true)
    all_both_correct  = np.array(all_both_correct)

    health_labels = {v: k.capitalize() for k, v in health_map.items()}

    print("\n" + "="*80)
    if cross_species_test:
        print(f"CROSS-SPECIES TEST RESULTS  |  Test Species: {test_species_name}")
        print(f"(Model trained on: {list(species_map.keys())}  —  {test_species_name} was NEVER seen in training)")
    else:
        print("COMPREHENSIVE TEST RESULTS")
    print("="*80)

    health_acc  = accuracy_score(all_health_true, all_health_preds)
    health_f1   = f1_score(all_health_true, all_health_preds, average='weighted', zero_division=0)
    health_prec = precision_score(all_health_true, all_health_preds, average='weighted', zero_division=0)
    health_rec  = recall_score(all_health_true, all_health_preds, average='weighted', zero_division=0)

    if cross_species_test:
        print(f"\n{'PRIMARY METRIC — HEALTH (cross-species generalisation)':^80}")
        print("-"*80)
        print(f"  Health Accuracy   : {health_acc:.4f} ({health_acc*100:.2f}%)")
        print(f"  Health F1 (w-avg) : {health_f1:.4f}")
        print(f"  Health Precision  : {health_prec:.4f}")
        print(f"  Health Recall     : {health_rec:.4f}")
        print(f"\n  [Species accuracy is NOT reported — {test_species_name} has no valid species label in training]")
    else:
        species_labels = {v: k.capitalize() for k, v in species_map.items()}
        species_acc = accuracy_score(all_species_true, all_species_preds)
        both_acc    = all_both_correct.mean()
        print(f"\n{'OVERALL ACCURACIES':^80}")
        print("-"*80)
        print(f"  Species Classification:  {species_acc:.4f} ({species_acc*100:.2f}%)")
        print(f"  Health Classification:   {health_acc:.4f} ({health_acc*100:.2f}%)")
        print(f"  Both Correct (Joint):    {both_acc:.4f} ({both_acc*100:.2f}%)")

    # Health Classification Report
    print(f"\n{'HEALTH/DISEASE CLASSIFICATION REPORT':^80}")
    print("-"*80)
    print(classification_report(
        all_health_true,
        all_health_preds,
        target_names=[health_labels[i] for i in sorted(health_labels.keys())],
        digits=4
    ))

    # Per health-class breakdown (grouped by true health label)
    if cross_species_test:
        print(f"\n{'PER-CLASS HEALTH ACCURACY  ({test_species_name} test set)':^80}")
        print("-"*80)
        for he_id in sorted(health_labels.keys()):
            mask = all_health_true == he_id
            if mask.sum() > 0:
                acc = (all_health_preds[mask] == he_id).mean()
                print(f"  {health_labels[he_id]:12s}: {acc:.4f} ({acc*100:.2f}%)  [{mask.sum():4d} samples]")

    # ── Health Confusion Matrix ───────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(8, 6))
    cm_health = confusion_matrix(all_health_true, all_health_preds)
    sns.heatmap(
        cm_health, annot=True, fmt='d', cmap='Greens', ax=ax,
        xticklabels=[health_labels[i] for i in sorted(health_labels.keys())],
        yticklabels=[health_labels[i] for i in sorted(health_labels.keys())]
    )
    title_suffix = f" ({test_species_name} — cross-species)" if cross_species_test else ""
    ax.set_title(f'Health/Disease Classification{title_suffix}\nAccuracy: {health_acc:.2%}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_health.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
    print(f"\nSaved health confusion matrix to 'confusion_matrix_health.png'")
    plt.close()

    if not cross_species_test:
        # Species Confusion Matrix (only meaningful for in-distribution test)
        species_labels = {v: k.capitalize() for k, v in species_map.items()}
        species_acc = accuracy_score(all_species_true, all_species_preds)
        fig, ax = plt.subplots(figsize=(8, 6))
        cm_species = confusion_matrix(all_species_true, all_species_preds)
        sns.heatmap(
            cm_species, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[species_labels[i] for i in sorted(species_labels.keys())],
            yticklabels=[species_labels[i] for i in sorted(species_labels.keys())]
        )
        ax.set_title(f'Species Classification\nAccuracy: {species_acc:.2%}',
                     fontsize=12, fontweight='bold')
        ax.set_ylabel('True Label')
        ax.set_xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig('confusion_matrix_species.png', dpi=CONFUSION_MATRIX_DPI, bbox_inches='tight')
        print(f"Saved species confusion matrix to 'confusion_matrix_species.png'")
        plt.close()

    print("\n" + "="*80)
    print("Testing complete!")
    print("="*80 + "\n")

    return {
        'health_accuracy':  health_acc,
        'health_f1':        health_f1,
        'health_precision': health_prec,
        'health_recall':    health_rec,
        'health_preds':     all_health_preds,
        'health_true':      all_health_true,
        'species_preds':    all_species_preds,
        'species_true':     all_species_true,
    }


## TRAINING


In [ ]:
results = train_model()
print(f"\n{'='*80}")
print("TRAINING COMPLETE  (Cross-Species: Trained Mango+Papaya, Tested Guava)")
print("="*80)
print(f"  Species Acc : N/A  (Guava never seen in training)")
print(f"  Health Acc  : {results['task_b_accuracy']*100:.2f}%")
print(f"  Health F1   : {results['health_f1']*100:.2f}%")
print(f"  Precision   : {results['health_precision']*100:.2f}%")
print(f"  Recall      : {results['health_recall']*100:.2f}%")


## COMPREHENSIVE TEST


In [ ]:
best_model = MultiTaskSwinLarge(num_species=NUM_SPECIES, num_health=NUM_HEALTH,
                          pretrained=PRETRAINED, dropout=DROPOUT).to(device)
ckpt = torch.load("best_model.pt", map_location=device)
best_model.load_state_dict(ckpt["model"])
print("Loaded best_model.pt")

print("\n" + "="*80)
print("Running Cross-Species Test Evaluation  (Test species: Guava)")
print("="*80 + "\n")

test_results = comprehensive_test(
    model=best_model,
    test_loader=test_loader,
    device=device,
    species_map=SPECIES_MAP,
    health_map=HEALTH_MAP,
    cross_species_test=True,
    test_species_name="Guava"
)


## SAMPLE PREDICTIONS


In [ ]:
print("\n" + "="*80)
print("Generating Sample Predictions Visualization")
print("="*80 + "\n")

# Label mappings for visualization
# Training species only — Guava is the test set (not shown in sample predictions)
species_labels = {
    0: 'Mango',
    1: 'Papaya'
}

health_labels = {
    0: 'Healthy',
    1: 'Anthracnose',
}

# Number of samples to display per class
amount = 10

# Collect samples from validation set
sample_images_by_class = {0: [], 1: []}
sample_predictions_by_class = {0: [], 1: []}
sample_ground_truth_by_class = {0: [], 1: []}

best_model.eval()
with torch.no_grad():
    for images, species_batch, health_batch in val_loader:
        images = images.to(device)
        
        outputs = best_model(images)
        species_preds = outputs[0].argmax(1)
        health_preds = outputs[1].argmax(1)
        
        for i in range(len(images)):
            species_class = species_batch[i].item()
            
            if len(sample_images_by_class[species_class]) < amount:
                sample_images_by_class[species_class].append(images[i].cpu())
                sample_predictions_by_class[species_class].append({
                    'species': species_preds[i].item(),
                    'health': health_preds[i].item()
                })
                sample_ground_truth_by_class[species_class].append({
                    'species': species_batch[i].item(),
                    'health': health_batch[i].item()
                })
        
        if all(len(samples) >= amount for samples in sample_images_by_class.values()):
            break

# Visualize
mean = np.array(NORMALIZE_MEAN)
std = np.array(NORMALIZE_STD)

fig, axes = plt.subplots(NUM_SPECIES, amount, figsize=(3*amount, 9))

for row, species_idx in enumerate(sorted(sample_images_by_class.keys())):
    for col in range(amount):
        ax = axes[row, col]
        
        img = sample_images_by_class[species_idx][col]
        pred = sample_predictions_by_class[species_idx][col]
        gt = sample_ground_truth_by_class[species_idx][col]
        
        # Denormalize and display
        img_display = img.numpy().transpose(1, 2, 0)
        img_display = std * img_display + mean
        img_display = np.clip(img_display, 0, 1)
        
        ax.imshow(img_display)
        ax.axis('off')
        
        # Check correctness
        both_correct = (pred['species'] == gt['species']) and (pred['health'] == gt['health'])
        
        # Create title
        pred_sp = species_labels[pred['species']]
        pred_he = health_labels[pred['health']]
        gt_sp = species_labels[gt['species']]
        gt_he = health_labels[gt['health']]
        
        title = f"Pred: {pred_sp}, {pred_he}\nTrue: {gt_sp}, {gt_he}"
        color = 'green' if both_correct else 'red'
        ax.set_title(title, fontsize=8, color=color, fontweight='bold')
    
    # Add species label
    fig.text(0.02, 0.5 + (1 - row) * 0.3, species_labels[species_idx],
             fontsize=12, fontweight='bold', va='center', rotation=90)

plt.suptitle(f'Sample Predictions - {amount} Samples per Class\n(Green=Correct, Red=Incorrect)',
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout(rect=[0.05, 0, 1, 0.99])
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
print(f"Saved sample predictions ({NUM_SPECIES*amount} total: {amount} per class) to 'sample_predictions.png'")
plt.close()


## FINAL SUMMARY

In [ ]:
print("\n" + "="*95)
print("FINAL RESULTS \u2014 Cross-Species Evaluation | Swin-Large + FPN + GCA")
print("  Trained on : Mango + Papaya   (85% train / 15% val)")
print("  Tested on  : Guava            (0% seen during training)")
print("="*95)
print(f"\n{'Metric':<28} {'Value':>10}")
print("-" * 42)
print(f"{'Health Acc  (Guava, unseen)':<28} {test_results['health_accuracy']*100:>9.2f}%")
print(f"{'Health F1   (w-avg)'        :<28} {test_results['health_f1']*100:>9.2f}%")
print(f"{'Health Precision'           :<28} {test_results['health_precision']*100:>9.2f}%")
print(f"{'Health Recall'              :<28} {test_results['health_recall']*100:>9.2f}%")
print("-" * 42)
print(f"  Species Acc (Task A) : N/A  — Guava was never seen during training")
print("="*95)

# Training-phase results (from train_model return value)
print(f"\n{'Training Phase (Mango+Papaya val set)'}")
print(f"{'Health Acc  (val, in-distribution)':<28} {results['task_b_accuracy']*100:>9.2f}%")
print(f"{'Health F1   (val, in-distribution)':<28} {results['health_f1']*100:>9.2f}%")


## TRAINING / VALIDATION LEARNING CURVES

In [ ]:
# ── Training / Validation Learning Curves ─────────────────────────────────
history_df = pd.read_csv("training_history.csv")
epochs = range(1, len(history_df) + 1)

# ── Figure 1: Loss ────────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(epochs, history_df["train_loss"], "b-o", markersize=3, label="Train Loss")
ax1.plot(epochs, history_df["val_loss"],   "r-o", markersize=3, label="Val Loss")
ax1.set_title("Training vs. Validation Loss", fontsize=13, fontweight="bold")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
ax1.legend(); ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("learning_curve_loss.png", dpi=150, bbox_inches="tight")
plt.show()
print("\u2705 Loss curve saved to 'learning_curve_loss.png'")

# ── Figure 2: Accuracy ────────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(8, 5))
ax2.plot(epochs, history_df["train_acc_health"],  "b-o",  markersize=3, label="Train Health Acc")
ax2.plot(epochs, history_df["val_acc_health"],    "r-o",  markersize=3, label="Val Health Acc")
ax2.plot(epochs, history_df["train_acc_species"], "b--s", markersize=3, label="Train Species Acc")
ax2.plot(epochs, history_df["val_acc_species"],   "r--s", markersize=3, label="Val Species Acc")
ax2.set_title("Training vs. Validation Accuracy", fontsize=13, fontweight="bold")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.legend(); ax2.grid(True, alpha=0.3); ax2.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("learning_curve_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()
print("\u2705 Accuracy curve saved to 'learning_curve_accuracy.png'")

## NORMALIZED CONFUSION MATRICES

In [ ]:
# ── Cross-Species Health Confusion Matrix (Guava test set) ───────────────────
sp_true  = test_results["species_true"]   # all dummy 0 for guava
sp_preds = test_results["species_preds"]
he_true  = test_results["health_true"]
he_preds = test_results["health_preds"]

_he_labels = ["Healthy", "Anthracnose"]

# 2x2 health confusion matrix — this is the only meaningful matrix for cross-species
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm_health     = confusion_matrix(he_true, he_preds, labels=[0, 1])
cm_health_norm = cm_health.astype(float) / cm_health.sum(axis=1, keepdims=True).clip(min=1)

sns.heatmap(cm_health, annot=True, fmt="d", cmap="Greens", ax=axes[0],
            xticklabels=_he_labels, yticklabels=_he_labels)
axes[0].set_title("Health Classification — Raw Counts\n(Guava test set — cross-species)",
                  fontsize=11, fontweight="bold")
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("True")

sns.heatmap(cm_health_norm, annot=True, fmt=".2f", cmap="Greens", ax=axes[1],
            xticklabels=_he_labels, yticklabels=_he_labels, vmin=0, vmax=1)
axes[1].set_title("Health Classification — Normalized\n(Guava test set — cross-species)",
                  fontsize=11, fontweight="bold")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("True")

plt.suptitle(
    "Swin-Large + FPN + GCA  |  Cross-Species Test: Guava\n"
    "(Model trained on Mango + Papaya — Guava never seen in training)",
    fontsize=12, fontweight="bold"
)
plt.tight_layout()
plt.savefig("confusion_matrix_cross_species.png", dpi=150, bbox_inches="tight")
plt.show()
print("\u2705 Cross-species health confusion matrix saved to 'confusion_matrix_cross_species.png'")

# Per health-class breakdown
print("\nPer-Class Health Accuracy (Guava — cross-species test):")
for he_id, he_name in enumerate(_he_labels):
    mask = he_true == he_id
    if mask.sum() > 0:
        acc = (he_preds[mask] == he_id).mean()
        print(f"  {he_name:12s}: {acc:.4f} ({acc*100:.2f}%)  [{mask.sum():4d} samples]")


## ROC AND PRECISION-RECALL CURVES

In [ ]:
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

# Re-run inference on guava test set to collect softmax probabilities
best_model.eval()
_he_probs_list = []
_he_true_roc   = []

with torch.no_grad():
    for imgs, _, y_he in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        _, logits_he = best_model(imgs)
        _he_probs_list.append(torch.softmax(logits_he, dim=1).cpu().numpy())
        _he_true_roc.extend(y_he.numpy())

he_probs_arr = np.vstack(_he_probs_list)
he_true_roc  = np.array(_he_true_roc)

# ── Health ROC ────────────────────────────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(7, 6))
fpr_h, tpr_h, _ = roc_curve(he_true_roc, he_probs_arr[:, 1])
auc_h = auc(fpr_h, tpr_h)
ax1.plot(fpr_h, tpr_h, "b-", lw=2, label=f"Anthracnose  (AUC = {auc_h:.4f})")
ax1.plot([0, 1], [0, 1], "k--", lw=1)
ax1.set_title("ROC Curve \u2014 Health Classification\n(Guava test set — cross-species)",
              fontsize=12, fontweight="bold")
ax1.set_xlabel("False Positive Rate"); ax1.set_ylabel("True Positive Rate")
ax1.legend(loc="lower right"); ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("roc_health.png", dpi=150, bbox_inches="tight")
plt.show()
print("\u2705 Saved: roc_health.png")

# ── Health Precision-Recall ───────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(7, 6))
prec_h, rec_h, _ = precision_recall_curve(he_true_roc, he_probs_arr[:, 1])
ap_h = average_precision_score(he_true_roc, he_probs_arr[:, 1])
ax2.plot(rec_h, prec_h, "b-", lw=2, label=f"Anthracnose  (AP = {ap_h:.4f})")
ax2.set_title("Precision-Recall Curve \u2014 Health\n(Guava test set — cross-species)",
              fontsize=12, fontweight="bold")
ax2.set_xlabel("Recall"); ax2.set_ylabel("Precision")
ax2.legend(); ax2.grid(True, alpha=0.3)
ax2.set_xlim(0, 1); ax2.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("pr_health.png", dpi=150, bbox_inches="tight")
plt.show()
print("\u2705 Saved: pr_health.png")

print("\nNote: Species ROC/PR curves are omitted — Guava (test species) was never")
print("seen in training and carries no valid species label for ROC computation.")


## ERROR ANALYSIS (HARD EXAMPLES)

In [ ]:
# # ── Collect misclassified examples from test set ───────────────────────────
# best_model.eval()
# _sp_names_err = {0: "Guava", 1: "Mango", 2: "Papaya"}
# _he_names_err = {0: "Healthy", 1: "Anthrac."}
# _mean = np.array(NORMALIZE_MEAN); _std = np.array(NORMALIZE_STD)

# from collections import defaultdict
# import itertools
# _buckets = defaultdict(list)

# with torch.no_grad():
#     for imgs, y_sp, y_he in test_loader:
#         imgs_dev = imgs.to(device, non_blocking=True)
#         logits_sp, logits_he = best_model(imgs_dev)
#         sp_prd = logits_sp.argmax(1).cpu()
#         he_prd = logits_he.argmax(1).cpu()
#         for i in range(len(imgs)):
#             if he_prd[i].item() != y_he[i].item() or sp_prd[i].item() != y_sp[i].item():
#                 _buckets[y_sp[i].item()].append({
#                     "img":     imgs[i].cpu(),
#                     "sp_true": y_sp[i].item(),  "he_true": y_he[i].item(),
#                     "sp_pred": sp_prd[i].item(), "he_pred": he_prd[i].item(),
#                     "he_wrong": he_prd[i].item() != y_he[i].item(),
#                     "sp_wrong": sp_prd[i].item() != y_sp[i].item(),
#                 })
#         if sum(len(v) for v in _buckets.values()) >= 15:
#             break

# # Pick up to 5 — one per species first for variety
# selected = []
# for sp_key in itertools.cycle([0, 1, 2]):
#     if len(selected) >= 5:
#         break
#     if _buckets[sp_key]:
#         selected.append(_buckets[sp_key].pop(0))

# n_show = len(selected)
# print(f"Displaying {n_show} misclassified examples "
#       f"({', '.join(_sp_names_err[e['sp_true']] for e in selected)})")

# fig, axes = plt.subplots(1, n_show, figsize=(n_show * 3.2, 3.8))
# if n_show == 1:
#     axes = [axes]

# for idx, ex in enumerate(selected):
#     img = ex["img"].numpy().transpose(1, 2, 0)
#     img = np.clip(_std * img + _mean, 0, 1)
#     axes[idx].imshow(img)
#     axes[idx].axis("off")
#     t   = f"True: {_sp_names_err[ex['sp_true']]}, {_he_names_err[ex['he_true']]}"
#     p   = f"Pred: {_sp_names_err[ex['sp_pred']]}, {_he_names_err[ex['he_pred']]}"
#     tag = []
#     if ex["he_wrong"]: tag.append("Health\u2717")
#     if ex["sp_wrong"]: tag.append("Species\u2717")
#     axes[idx].set_title(f"{t}\n{p}\n({', '.join(tag)})",
#                         fontsize=8, color="red", fontweight="bold")

# plt.tight_layout()
# plt.savefig("error_analysis.png", dpi=150, bbox_inches="tight")
# plt.show()
# print("\u2705 Error analysis grid saved to 'error_analysis.png'")

## FEATURE SPACE — t-SNE PROJECTION

In [ ]:
from sklearn.manifold import TSNE

# Extract penultimate-layer embeddings via a forward hook on the dropout layer
_embeddings, _lbls_he, _lbls_correct = [], [], []

def _hook_fn(module, inp, out):
    _embeddings.append(out.detach().cpu())

_hook = best_model.dropout.register_forward_hook(_hook_fn)

best_model.eval()
with torch.no_grad():
    for imgs, _, y_he in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        _, logits_he = best_model(imgs)
        he_preds_tsne = logits_he.argmax(1).cpu()
        _lbls_he.extend(y_he.numpy())
        _lbls_correct.extend((he_preds_tsne == y_he).numpy().astype(int))
        best_model(imgs)  # second pass to trigger hook (first already triggered above)

_hook.remove()

# Use only every-other embedding (hook fires once per forward call, we called twice)
all_emb = torch.cat(_embeddings[::2], dim=0).numpy()
lbls_he      = np.array(_lbls_he)
lbls_correct = np.array(_lbls_correct)

print(f"Embeddings shape : {all_emb.shape}")
print("Running t-SNE (may take ~30-90 s) ...")

tsne   = TSNE(n_components=2, perplexity=30, max_iter=1000,
              random_state=SEED, init="pca", learning_rate="auto")
emb_2d = tsne.fit_transform(all_emb)
print("t-SNE complete.")

_he_color_map   = {0: "forestgreen",  1: "crimson"}
_he_names_tsne  = {0: "Healthy",      1: "Anthracnose"}
_ok_color_map   = {1: "royalblue",    0: "darkorange"}
_ok_names_tsne  = {1: "Correct",      0: "Incorrect"}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: coloured by health status (primary metric)
for he_id, color in _he_color_map.items():
    mask = lbls_he == he_id
    axes[0].scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                    c=color, label=_he_names_tsne[he_id], alpha=0.65, s=18, edgecolors="none")
axes[0].set_title("t-SNE \u2014 Coloured by Health Status\n(Guava test set)", fontsize=12, fontweight="bold")
axes[0].legend(fontsize=10); axes[0].set_xlabel("t-SNE 1"); axes[0].set_ylabel("t-SNE 2")
axes[0].grid(True, alpha=0.2)

# Right: coloured by prediction correctness (more informative for cross-species)
for ok_id, color in _ok_color_map.items():
    mask = lbls_correct == ok_id
    axes[1].scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                    c=color, label=_ok_names_tsne[ok_id], alpha=0.65, s=18, edgecolors="none")
axes[1].set_title("t-SNE \u2014 Correct vs Incorrect Prediction\n(cross-species generalisation)",
                  fontsize=12, fontweight="bold")
axes[1].legend(fontsize=10); axes[1].set_xlabel("t-SNE 1"); axes[1].set_ylabel("t-SNE 2")
axes[1].grid(True, alpha=0.2)

plt.suptitle(
    "t-SNE: Penultimate Layer Features | Swin-Large + FPN + GCA\n"
    "Test set: Guava (trained on Mango + Papaya)",
    fontsize=13, fontweight="bold"
)
plt.tight_layout()
plt.savefig("tsne_projection.png", dpi=150, bbox_inches="tight")
plt.show()
print("\u2705 t-SNE projection saved to 'tsne_projection.png'")


## COMPUTATIONAL EFFICIENCY METRICS

In [ ]:
import time, os

# ── Parameter count ────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in best_model.parameters())
trainable_params = sum(p.numel() for p in best_model.parameters() if p.requires_grad)

# ── Model file size ────────────────────────────────────────────────────────
model_size_mb = os.path.getsize("best_model.pt") / 1e6

# ── FLOPs (via thop or fvcore) ────────────────────────────────────────────
flops_gflops = None
_dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
try:
    from thop import profile as _thop_profile
    best_model.eval()
    with torch.no_grad():
        _macs, _ = _thop_profile(best_model, inputs=(_dummy,), verbose=False)
    flops_gflops = 2 * _macs / 1e9
    print(f"FLOPs (thop): {flops_gflops:.2f} GFLOPs")
except Exception as _e1:
    try:
        from fvcore.nn import FlopCountAnalysis
        best_model.eval()
        _fc = FlopCountAnalysis(best_model, _dummy)
        flops_gflops = _fc.total() / 1e9
        print(f"FLOPs (fvcore): {flops_gflops:.2f} GFLOPs")
    except Exception as _e2:
        print(f"FLOPs unavailable (install thop or fvcore): {_e2}")

# ── Inference latency and throughput ──────────────────────────────────────
best_model.eval()
_warmup = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    for _ in range(20):  # warm-up
        best_model(_warmup)
if device.type == "cuda":
    torch.cuda.synchronize()

_N = 100
_latencies = []
with torch.no_grad():
    for _ in range(_N):
        _t0 = time.perf_counter()
        best_model(_warmup)
        if device.type == "cuda":
            torch.cuda.synchronize()
        _latencies.append((time.perf_counter() - _t0) * 1000)

mean_lat_ms = float(np.mean(_latencies))
std_lat_ms  = float(np.std(_latencies))
throughput  = 1000.0 / mean_lat_ms

# ── Print summary table ────────────────────────────────────────────────────
print("\n" + "="*65)
print("COMPUTATIONAL EFFICIENCY METRICS — Swin-Large + FPN + GCA")
print("="*65)
rows = [
    ("Total Parameters",         f"{total_params/1e6:.2f} M"),
    ("Trainable Parameters",      f"{trainable_params/1e6:.2f} M"),
    ("FLOPs",                     f"{flops_gflops:.2f} GFLOPs" if flops_gflops else "N/A"),
    ("Inference Latency (mean)",  f"{mean_lat_ms:.2f} ms"),
    ("Inference Latency (std)",   f"{std_lat_ms:.2f} ms"),
    ("Throughput",                f"{throughput:.1f} img/s"),
    ("Model Size",                f"{model_size_mb:.2f} MB"),
    ("Hardware",                  str(device)),
]
for metric, value in rows:
    print(f"  {metric:<32} {value:>18}")
print("="*65)

# ── Bar chart ─────────────────────────────────────────────────────────────
_bar_data = {
    "Params\n(M)":      total_params / 1e6,
    "Latency\n(ms)":   mean_lat_ms,
    "Throughput\n(/10 FPS)": throughput / 10,
    "Model Size\n(MB)": model_size_mb,
}
if flops_gflops:
    _bar_data["GFLOPs"] = flops_gflops

_bar_colors = ["royalblue", "tomato", "forestgreen", "darkorange", "purple"]
fig, ax = plt.subplots(figsize=(10, 5))
_bars = ax.bar(list(_bar_data.keys()), list(_bar_data.values()),
               color=_bar_colors[:len(_bar_data)], edgecolor="white", linewidth=0.8)
for _b, _v in zip(_bars, _bar_data.values()):
    ax.text(_b.get_x() + _b.get_width() / 2, _b.get_height() + max(_bar_data.values()) * 0.01,
            f"{_v:.1f}", ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_title("Computational Efficiency — Swin-Large + FPN + GCA",
             fontsize=13, fontweight="bold")
ax.set_ylabel("Value"); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("efficiency_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("\u2705 Efficiency metrics chart saved to 'efficiency_metrics.png'")


## EXPLAINABLE AI — GradCAM++ & LIME

In [ ]:
# ── Install lime if needed ─────────────────────────────────────────────────
import subprocess, sys as _sys
try:
    from lime import lime_image as _lime_check
except ImportError:
    print("Installing lime ...")
    subprocess.check_call([_sys.executable, "-m", "pip", "install", "lime", "-q"])
    print("\u2705 lime installed")

import cv2
from lime import lime_image
from skimage.segmentation import mark_boundaries


# ─────────────────────────────────────────────────────────────────────────────
# Grad-CAM++ implementation for MultiTaskSwinLarge
# Target: model.feature_fusion[3]  (last Conv2d → (B, 256, 7, 7))
# ─────────────────────────────────────────────────────────────────────────────

class GradCAMPlusPlus:
    """Grad-CAM++ for the multi-task Swin-Large model."""

    def __init__(self, model, target_layer):
        self.model       = model
        self.activations = None
        self.gradients   = None
        target_layer.register_forward_hook(self._save_act)
        target_layer.register_full_backward_hook(self._save_grad)

    def _save_act(self, module, inp, out):
        self.activations = out.detach()

    def _save_grad(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, img_tensor, task="health"):
        """
        Returns a numpy CAM array [H, W] normalised to [0, 1].
        task = "health" (2-class) | "species" (3-class)
        """
        self.model.eval()
        self.model.zero_grad()

        with torch.enable_grad():
            logits_sp, logits_he = self.model(img_tensor)
            logits    = logits_he if task == "health" else logits_sp
            class_idx = logits.argmax(dim=1).item()
            logits[0, class_idx].backward()

        grads = self.gradients[0]    # (C, H, W)
        acts  = self.activations[0]  # (C, H, W)

        # Grad-CAM++ alpha weights
        alpha_num   = grads.pow(2)
        alpha_denom = (2 * grads.pow(2) +
                       (acts * grads.pow(3)).sum(dim=(1, 2), keepdim=True))
        alpha   = alpha_num / (alpha_denom + 1e-8)
        weights = F.relu((alpha * grads).sum(dim=(1, 2), keepdim=True))  # (C,1,1)
        cam     = F.relu((weights * acts).sum(dim=0))                    # (H, W)

        # Upsample to input resolution
        H, W = img_tensor.shape[-2:]
        cam = F.interpolate(
            cam.unsqueeze(0).unsqueeze(0).float(),
            size=(H, W), mode="bilinear", align_corners=False
        ).squeeze().cpu().numpy()

        cam -= cam.min()
        cam /= cam.max() + 1e-8
        return cam


def _cam_overlay(img_rgb_uint8, cam, alpha=0.5):
    """Blend jet heatmap over the original uint8 RGB image."""
    hmap = (plt.cm.jet(cam)[:, :, :3] * 255).astype(np.uint8)
    return cv2.addWeighted(img_rgb_uint8, 1 - alpha, hmap, alpha, 0)


def _lime_explain(model, img_rgb_uint8, device, num_samples=1000, task="health"):
    """Run LIME and return boundary-marked image (float [0, 1] RGB)."""
    _mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    _std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def _predict(images):
        tensors = [
            (torch.from_numpy(img).float().permute(2, 0, 1) / 255.0 - _mean) / _std
            for img in images
        ]
        batch = torch.stack(tensors).to(device)
        with torch.no_grad():
            ls, lh = model(batch)
            logits = lh if task == "health" else ls
            return torch.softmax(logits, dim=1).cpu().numpy()

    exp = lime_image.LimeImageExplainer().explain_instance(
        img_rgb_uint8, _predict,
        top_labels=1, hide_color=0,
        num_samples=num_samples, batch_size=32
    )
    top_label = exp.top_labels[0]
    _, mask = exp.get_image_and_mask(
        top_label, positive_only=False,
        num_features=10, hide_rest=False
    )
    return mark_boundaries(img_rgb_uint8 / 255.0, mask,
                           color=(0, 1, 0), mode="thick")


# ─────────────────────────────────────────────────────────────────────────────
# Main visualization
# ─────────────────────────────────────────────────────────────────────────────

def run_xai(model, test_df, device,
            n_per_class=1, num_lime_samples=1000):
    """
    4-column figure per sample:
      (a) Original  (b) GradCAM++ Overlay
      (c) GradCAM++ Heatmap  (d) LIME Decision Boundaries
    """
    # Hook onto last Conv2d in feature_fusion (index 3 → (B, 256, 7, 7))
    # Using pre-ReLU conv output avoids inplace-op issues with autograd.
    target_layer = model.feature_fusion[3]
    gradcam = GradCAMPlusPlus(model, target_layer)

    _mean_t = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    _std_t  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    _he_map = {0: "Healthy", 1: "Anthracnose"}

    # Select n_per_class samples per health class from test_df
    samples = []
    for cls in ["Healthy", "Anthracnose"]:
        sub = test_df[test_df["class"] == cls]
        for _, row in sub.sample(n=min(n_per_class, len(sub)),
                                  random_state=SEED).iterrows():
            samples.append({"path":  row["image_path"],
                             "plant": row["plant"],
                             "cls":   row["class"]})

    n = len(samples)
    fig, axes = plt.subplots(n, 4, figsize=(16, 4.8 * n))
    if n == 1:
        axes = axes[np.newaxis, :]

    last_im = None

    for row_i, sample in enumerate(samples):
        print(f"  [{row_i+1}/{n}] {sample['plant']} — {sample['cls']}")

        # ── Load & resize ─────────────────────────────────────────────────
        bgr = cv2.imread(sample["path"])
        img = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (224, 224))       # uint8 [0, 255]

        # ── Normalised tensor ─────────────────────────────────────────────
        t = ((torch.from_numpy(img).float().permute(2, 0, 1) / 255.0
              - _mean_t) / _std_t).unsqueeze(0).to(device)

        # ── Health prediction ─────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            _, lh   = model(t)
            probs   = torch.softmax(lh, dim=1)[0]
            he_pred = probs.argmax().item()
            he_conf = probs.max().item()

        # ── GradCAM++ ─────────────────────────────────────────────────────
        cam     = gradcam.generate(t, task="health")
        overlay = _cam_overlay(img, cam, alpha=0.5)

        # ── LIME ──────────────────────────────────────────────────────────
        print(f"    LIME ({num_lime_samples} samples) ...")
        lime_img = _lime_explain(model, img, device, num_lime_samples)

        # ── Row label ─────────────────────────────────────────────────────
        row_title = (f"{sample['plant']}\n"
                     f"True : {sample['cls']}\n"
                     f"Pred : {_he_map[he_pred]}")

        # ── Plot ──────────────────────────────────────────────────────────
        axes[row_i, 0].imshow(img)
        axes[row_i, 0].set_title("(a) Original Image",
                                  fontsize=11, fontweight="bold")
        axes[row_i, 0].axis("off")
        axes[row_i, 0].text(-0.08, 0.5, row_title,
                             transform=axes[row_i, 0].transAxes,
                             ha="right", va="center",
                             fontsize=9, fontweight="bold",
                             bbox=dict(boxstyle="round,pad=0.3",
                                       facecolor="wheat", alpha=0.7))

        axes[row_i, 1].imshow(overlay)
        axes[row_i, 1].set_title("(b) GradCAM++ Overlay",
                                  fontsize=11, fontweight="bold")
        axes[row_i, 1].axis("off")

        last_im = axes[row_i, 2].imshow(cam, cmap="jet", vmin=0, vmax=1)
        axes[row_i, 2].set_title("(c) GradCAM++ Heatmap",
                                  fontsize=11, fontweight="bold")
        axes[row_i, 2].axis("off")

        axes[row_i, 3].imshow(lime_img)
        axes[row_i, 3].set_title("(d) LIME Decision Boundaries",
                                  fontsize=11, fontweight="bold")
        axes[row_i, 3].axis("off")

    # ── Colorbar ──────────────────────────────────────────────────────────
    fig.subplots_adjust(left=0.12, right=0.88, hspace=0.05)
    cax = fig.add_axes([0.90, 0.15, 0.015, 0.70])
    cb  = fig.colorbar(last_im, cax=cax)
    cb.set_label("Attention Intensity", rotation=270,
                 labelpad=18, fontsize=10)

    plt.savefig("xai_gradcam_lime.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("\u2705 XAI visualization saved to 'xai_gradcam_lime.png'")


# ─────────────────────────────────────────────────────────────────────────────
# Execute
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("EXPLAINABLE AI — GradCAM++ & LIME")
print("=" * 70)
print("Selecting 1 Healthy + 1 Anthracnose sample from the test set\n")

run_xai(
    model=best_model,
    test_df=test_df,
    device=device,
    n_per_class=1,
    num_lime_samples= 1000
)

print("\n" + "=" * 70)
print("\u2705 XAI ANALYSIS COMPLETE")
print("=" * 70)
